### Теория. Объекты-сущности, Объекты-значения и внедренные объекты
---

#### 1. Что такое объекты‑сущности

**Объект‑сущность** `entity object` — это объект, который в программе моделирует реальный или абстрактный объект предметной области и главным образом идентифицируется не по своим атрибутам, а по уникальному идентификатору `(ID)`.
`

**Почему `ID` важнее имени**

В предметной области (бизнес‑логике) **сущность** — это «кто/что это», а не «как это выглядит сейчас».

- `id` — это идентичность (кто это).
- `name`, `email`, `age` и т. п. — это атрибуты (какие у него текущие свойства).

Атрибуты могут меняться: человек сменил имя, переехал, сменил телефон. Но он остаётся тем же самым человеком. Поэтому в коде мы сравниваем сущности по `ID`.

```python
user1 = User(1, "John")
user2 = User(1, "Jon")  # имя изменилось

print(user1 == user2)  # True, потому что это всё ещё тот же пользователь (id=1)
```

#### Как это выражается в Python: `__eq__` и `__hash__`
---

1. `__eq__` — это «магический» `(dunder)` метод в `Python`, который определяет поведение оператора `==` для объектов класса.

**Что он делает**

- Когда пишешь `a == b`, `Python` на самом деле вызывает `a.__eq__(b)`. 
- Для встроенных типов всё уже реализовано: числа сравниваются по значению, списки — по элементам и т. д.
- Для своих классов можно задать свою логику равенства, т.е. переопределить метод `__eq__`

```python
class User:
    def __init__(self, id, name):
        self.id = id
        self.name = name

    def __eq__(self, other):
        # Сравниваем только по id: пользователи равны, если у них одинаковый ID
        if not isinstance(other, User):
            return NotImplemented
        return self.id == other.id
```

#### Теперь:

```python
u1 = User(1, "Alice")
u2 = User(1, "Bob")
u3 = User(2, "Alice")

print(u1 == u2)  # True  (одинаковый id)
print(u1 == u3)  # False (разные id)
```

#### Зачем возвращать NotImplemented

#### Строка:

```python
if not isinstance(other, User):
    return NotImplemented
```
нужна, чтобы корректно работать с объектами других типов.

- Если вернуть `False, `Python` решит, что объекты точно не равны, и не будет пробовать другие варианты.
- Если вернуть `NotImplemented`, `Python` попробует вызвать `other.__eq__(self)` (если у other есть такой метод) или скажет, что сравнение не поддерживается.

#### Пример:

```python
class MyNum:
    def __init__(self, x):
        self.x = x

    def __eq__(self, other):
        if isinstance(other, (int, float)):
            return self.x == other
        if isinstance(other, MyNum):
            return self.x == other.x
        return NotImplemented

n = MyNum(5)
print(n == 5)      # True
print(5 == n)      # Тоже True: Python вызовет n.__eq__(5), а там логика поддерживает int
```

Если бы мы вместо `NotImplemented` вернули `False` при несовпадении типов, `5 == n` всегда было бы `False`, даже если мы хотим поддерживать такое сравнение.

----

### 2. Объекты‑значения (Value Objects, VO)

- У объекта‑значения нет собственной идентичности: важны только его свойства (значения). 
- Если два объекта‑значения имеют одинаковые поля — они считаются равными.

#### Примеры

- **Деньги:** `Money(100, "RUB")` и другой `Money(100, "RUB")` — это одно и то же значение. 
    - *Неважно, `«какой экземпляр»`, важно, сколько и в какой валюте.*

- **Диапазон дат**: `DateRange(start=2025-01-01, end=2025-01-31)`. 
    - *Если у двух таких объектов одинаковые даты — они равны.*

- **Адрес:** `Address(city="Москва", street="Ленина", house="1")`
    - если все поля совпали, это тот же адрес.

Для сущностей мы сравниваем по `ID`, для `VO` — по содержимому.

#### Ключевые свойства `Value Objects`

- Неизменяемость `(immutability)`. После создания `VO` не должен меняться. Вместо «изменить сумму» создают новый объект с новой суммой. Это сильно упрощает рассуждение о коде и делает `VO` безопасными в многопоточности.

- Равенство по значению. Два `VO` равны, если равны все их поля.

- Отсутствие жизненного цикла и `ID`. У VO нет понятия «создали, обновили, удалили». Это просто значение.

- Самодостаточность и выразительность. VO инкапсулирует логику, связанную с этим значением: валидацию, преобразования, арифметику. 
    - Например, `Money` умеет складывать только деньги одной валюты, а `DateRange` может проверять, пересекается ли он с другим диапазоном.

#### Простой пример с @dataclass
```python

from dataclasses import dataclass

@dataclass(frozen=True)  # frozen=True делает объект неизменяемым
class Money:
    amount: float
    currency: str

    def __post_init__(self):
        if self.amount < 0:
            raise ValueError("amount must be non-negative")
        if not self.currency:
            raise ValueError("currency is required")
```

#### Теперь:

```python

m1 = Money(100.0, "RUB")
m2 = Money(100.0, "RUB")
print(m1 == m2)          # True (сравниваются поля)
# m1.amount = 200        # Ошибка: объект заморожен (frozen)
```

- `frozen=True` даёт неизменяемость,
- `dataclass` автоматически генерирует `__eq__` по полям — как раз то, что нужно для VO.

#### Более «DDD‑шный» пример: диапазон дат
```python
from dataclasses import dataclass
from datetime import date

@dataclass(frozen=True)
class DateRange:
    start: date
    end: date

    def __post_init__(self):
        if self.start > self.end:
            raise ValueError("start must be <= end")

    def overlaps(self, other: "DateRange") -> bool:
        return self.start <= other.end and other.start <= self.end
```

Здесь VO не просто хранит данные, а несёт доменную логику: проверку корректности и проверку пересечения диапазонов.

---



### 3.Встраиваемые объекты `Embedded Objects` 


Здесь `embedded` означает, что один объект «живёт внутри» другого и не имеет самостоятельной жизни вне него. 
В разных языках и подходах это называют по‑разному: `composition`, `embedded object`, `value object`, `component`.

#### Что это значит на практике

**Пример, Address как часть User.**

```python
class Address:
    def __init__(self, street, house, zipcode):
        self.street = street
        self.house = house
        self.zipcode = zipcode

    def __str__(self):
        return f"{self.street}, {self.house}, {self.zipcode}"

class User:
    def __init__(self, id, name, address: Address):
        self.id = id
        self.name = name
        self.address = address  # Address — встраиваемый объект
```
> *Запись `address: Address` — это аннотация типа `(type hint)` для аргумента функции/метода в `Python`.*
> - *`address`— имя параметра (переменной внутри метода).*
> - *`Address` — ожидаемый тип этого параметра (класс, экземпляр которого должны передать)**.
> - *Двоеточие : просто отделяет имя параметра от указания типа*.
>
> *То есть фраза читается как: «параметр address должен быть объектом типа Address».*


Здесь `address` — это не просто «пара полей в User», а полноценный объект, который:

- Не имеет собственного жизненного цикла вне `User`. 
- Обычно его создают вместе с пользователем и удаляют вместе с ним.
- Может быть общим для нескольких сущностей (если это `value object`), но чаще он «принадлежит» одному владельцу.
- Инкапсулирует логику, относящуюся к адресу (форматирование, валидация, нормализация).

**Зачем так делают**

- **Чистота модели**: адрес — это отдельная концепция, и у неё может быть своя логика.
- **Переиспользование:** класс `Address` можно использовать и в `Order`, и в `Company`, и т. д.
- **Читаемость сигнатур:** `def __init__(self, address: Address)` сразу говорит, что ожидается полноценный адрес, а не набор отдельных аргументов.
- **Валидация и инварианты:** можно гарантировать, что адрес всегда находится в корректном состоянии (например, через `raise ValueError`).

----


### Задача

В данном упражнении необходимо реализовать класс `Url`, который позволяет извлекать из `HTTP адреса`, представленного строкой, его части.

**Класс** должен содержать методы:

- `__init__` — принимает на вход `HTTP адрес` в виде строки.

- `get_scheme()` — возвращает протокол передачи данных.

- `get_hostname()` — возвращает имя хоста.

- `get_query_params()` — возвращает параметры запроса в виде пар ключ-значение объекта.

- `get_query_param()` — получает значение параметра запроса по имени. Если параметр с переданным именем не существует, метод возвращает значение заданное вторым параметром (по умолчанию равно `None`).

- `__eq__(self, other)` — сравнивает два объекта класса `Url` и возвращает результат сравнения — `True` или `False`.

```python
url = Url('http://hexlet.io:80?key=value&key2=value2')
url.get_scheme() # http
url.get_hostname() # hexlet.io
url.get_query_params()
# {
#  key: [value],
#  key2: [value2],
# }
url.get_query_param('key') # value
# второй параметр — значение по умолчанию
url.get_query_param('key2', 'lala') # value2
url.get_query_param('new', 'ehu') # ehu
url.get_query_param('new') # None
url == Url('http://hexlet.io:80?key=value&key2=value2') # True
url == Url('http://hexlet.io:80?key=value') # False
```

#### Подсказки

- для парсинга `url` воспользуйтесь функциями `urlparse` и `parse_qs` из модуля [urllib](https://docs-python.ru/standart-library/modul-urllib-parse-python/)

- __eq__
